Qdrant Vectordb Ingestion

In [1]:
import os
from dotenv import load_dotenv
import json 
load_dotenv()

False

In [2]:
!pip install -q langchain_qdrant langchain_huggingface sentence-transformers

In [3]:
from typing import List, Dict
from tqdm import tqdm
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore


In [4]:
QDRANT_URL ="https://a8673c43-e709-40d5-b254-4900c45d634f.us-east-1-1.aws.cloud.qdrant.io:6333"
QDRANT_API_KEY = ""

In [17]:
EMBEDDING_MODELS = {
    "MedEmbed-base-v0.1": "abhinand/MedEmbed-base-v0.1",
    "pubmedbert-base-embeddings": "NeuML/pubmedbert-base-embeddings",
    "bge-m3": "BAAI/bge-m3",
    "Bio_ClinicalBERT": "emilyalsentzer/Bio_ClinicalBERT",
    "e5-large-v2": "intfloat/e5-large-v2",
    "e5-base-v2": "intfloat/e5-base-v2",
    "all-mini": "sentence-transformers/all-MiniLM-L6-v2"
}

In [6]:
from pathlib import Path

def load_all_chunks(chunks_dir: Path) -> List[Document]:
    """
    Load all JSON chunk files from the processed_chunks directory
    and convert them to LangChain Document objects.
    
    Args:
        chunks_dir: Path to the directory containing JSON chunk files
        
    Returns:
        List of LangChain Document objects
    """
    chunks_dir = Path(chunks_dir)
    all_documents = []
    json_files = list(chunks_dir.glob("*.json"))
    
    print(f"\nFound {len(json_files)} JSON files to process")
    
    for json_file in tqdm(json_files, desc="Loading chunks"):
        try:
            with open(json_file, "r", encoding="utf-8") as f:
                data = json.load(f)
            
            # Convert dictionaries to LangChain Document objects
            documents = [
                Document(
                    page_content=chunk["page_content"],
                    metadata=chunk["metadata"]
                )
                for chunk in data
            ]
            
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} chunks from {json_file.name}")
            
        except Exception as e:
            print(f"Error loading {json_file.name}: {e}")
            continue
    
    print(f"\n✅ Total documents loaded: {len(all_documents)}")
    return all_documents

In [7]:
def create_embeddings(model_name: str, device: str = "cpu") -> HuggingFaceEmbeddings:
    """
    Create HuggingFace embeddings for a given model.
    
    Args:
        model_name: HuggingFace model identifier
        device: Device to run the model on ('cpu' or 'cuda')
        
    Returns:
        HuggingFaceEmbeddings instance
    """
    print(f"  🔧 Initializing embeddings: {model_name}")
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={'device': 'cuda' if device == 'cuda' else 'cpu'},
        encode_kwargs={'normalize_embeddings': True}  # Normalize for better similarity search
    )


In [8]:
def ingest_to_qdrant(documents: List[Document], embeddings: HuggingFaceEmbeddings, collection_name: str,
    qdrant_url: str = QDRANT_URL, api_key: str = None, force_recreate: bool = False
    ) -> QdrantVectorStore:
    
    """
    Ingest documents into Qdrant with the specified embeddings.
    
    Args:
        documents: List of LangChain Document objects
        embeddings: HuggingFaceEmbeddings instance
        collection_name: Name of the Qdrant collection
        qdrant_url: URL of the Qdrant instance
        api_key: API key for Qdrant Cloud (optional for local)
        force_recreate: Whether to recreate the collection if it exists
        
    Returns:
        QdrantVectorStore instance
    """
    print(f"Ingesting {len(documents)} documents into collection: {collection_name}")
    
    try:
        # Prepare connection parameters
        connection_params = {
            "documents": documents,
            "embedding": embeddings,
            "url": qdrant_url,
            "collection_name": collection_name,
            "force_recreate": force_recreate
        }
        
        # Add API key if provided (for Qdrant Cloud)
        if api_key:
            connection_params["api_key"] = api_key
        
        vector_store = QdrantVectorStore.from_documents(**connection_params)
        print(f"  ✅ Successfully created collection: {collection_name}")
        return vector_store
        
    except Exception as e:
        print(f"  ✗ Error ingesting to {collection_name}: {e}")
        raise


In [9]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Access the processed_chunks folder
chunks_dir = '/content/drive/My Drive/processed_chunks'

# Verify the folder exists
if os.path.exists(chunks_dir):
    print(f"✅ Successfully accessed: {chunks_dir}")
    print(f"Contents: {os.listdir(chunks_dir)}")
else:
    print(f"✗ Folder not found at {chunks_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Successfully accessed: /content/drive/My Drive/processed_chunks
Contents: ['ada_cardio_disease_manag_chunks.json', 'ada_chapter13_chunks.json', 'ada_chapter15_chunks.json', 'ada_chapter17_chunks.json', 'ada_chapter2_chunks.json', 'ada_chapter3_chunks.json', 'ada_chapter4_chunks.json', 'ada_chapter7_chunks.json', 'ada_children_adolescents_standards_chunks.json', 'ada_chronic_kidney_manag_chunks.json', 'ada_fascilitating_pos_behaviors_chunks.json', 'ada_glycemic_hypoglycemia_chunks.json', 'ada_improv_care_chunks.json', 'ada_obesity_and_weight_chunks.json', 'ada_pharma_approaches_chunks.json', 'ada_retino_neuro_footcare_chunks.json']


Load all processed chunks

In [10]:
print("\n Loading all processed chunks...")
all_documents = load_all_chunks(chunks_dir)


 Loading all processed chunks...

Found 16 JSON files to process



 Loading all processed chunks...

Found 16 JSON files to process


Loading chunks:   0%|          | 0/16 [00:00<?, ?it/s]


 Loading all processed chunks...

Found 16 JSON files to process


Loading chunks:   0%|          | 0/16 [00:00<?, ?it/s]

Loaded 221 chunks from ada_cardio_disease_manag_chunks.json
Loaded 152 chunks from ada_chapter13_chunks.json
Loaded 147 chunks from ada_chapter15_chunks.json
Loaded 15 chunks from ada_chapter17_chunks.json
Loaded 200 chunks from ada_chapter2_chunks.json
Loaded 88 chunks from ada_chapter3_chunks.json
Loaded 211 chunks from ada_chapter4_chunks.json
Loaded 130 chunks from ada_chapter7_chunks.json
Loaded 169 chunks from ada_children_adolescents_standards_chunks.json
Loaded 112 chunks from ada_chronic_kidney_manag_chunks.json
Loaded 393 chunks from ada_fascilitating_pos_behaviors_chunks.json
Loaded 132 chunks from ada_glycemic_hypoglycemia_chunks.json
Loaded 110 chunks from ada_improv_care_chunks.json


Loading chunks:  88%|████████▊ | 14/16 [00:00<00:00, 62.72it/s]

Loaded 139 chunks from ada_obesity_and_weight_chunks.json


Loading chunks: 100%|██████████| 16/16 [00:00<00:00, 67.68it/s]

Loaded 284 chunks from ada_pharma_approaches_chunks.json
Loaded 129 chunks from ada_retino_neuro_footcare_chunks.json

✅ Total documents loaded: 2632


In [11]:
import torch

# Detect GPU with detailed info
print("GPU Detection:")
print(f"  CUDA available: {torch.cuda.is_available()}")
print(f"  CUDA device count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"  Primary device: {torch.cuda.get_device_name(0)}")
    device = 'cuda'
    print(f"✅ Using GPU: {device}\n")
else:
    device = 'cpu'
    print(f"⚠️ Using CPU: {device}\n")

GPU Detection:
  CUDA available: True
  CUDA device count: 1
  Primary device: Tesla T4
✅ Using GPU: cuda



In [12]:
# Verify sentence-transformers is imported
try:
    from sentence_transformers import SentenceTransformer
    print("✅ sentence-transformers imported successfully")
except ImportError:
    print("⚠️ Restarting kernel to load sentence-transformers...")
    import os
    os.kill(os.getpid(), 9)

✅ sentence-transformers imported successfully


In [15]:
HF_TOKEN=""


In [18]:
results = {}

for model_key, model_path in EMBEDDING_MODELS.items():
    print(f"Processing Model: {model_key}")
    try:
        # Create embeddings
        embeddings = create_embeddings(model_path, device=device)
        
        # Create collection name (sanitize for Qdrant)
        collection_name = f"ada_model_{model_key.lower().replace('-', '_')}"
        
        # Ingest to Qdrant
        vector_store = ingest_to_qdrant(
            documents=all_documents,
            embeddings=embeddings,
            collection_name=collection_name,
            api_key=QDRANT_API_KEY,
            force_recreate=False  # Set to True to overwrite existing collections
        )
        
        results[model_key] = {
            "collection_name": collection_name,
            "status": "success",
            "document_count": len(all_documents)
        }
        print(f"✅ Successfully processed {model_key}\n")
        
    except Exception as e:
        print(f"❌ Failed to process {model_key}: {e}\n")
        results[model_key] = {
            "collection_name": f"ada_model_{model_key.lower().replace('-', '_')}",
            "status": "failed",
            "error": str(e)
        }

# Summary
print("\n" + "="*60)
print("INGESTION SUMMARY")
print("="*60)
for model, result in results.items():
    status = result["status"]
    print(f"{model}: {status}")

Processing Model: MedEmbed-base-v0.1
  🔧 Initializing embeddings: abhinand/MedEmbed-base-v0.1


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ingesting 2632 documents into collection: ada_model_all_mini
  ✅ Successfully created collection: ada_model_all_mini
✅ Successfully processed all-mini


INGESTION SUMMARY
MedEmbed-base-v0.1: success
pubmedbert-base-embeddings: success
bge-m3: success
Bio_ClinicalBERT: success
e5-large-v2: success
e5-base-v2: success
all-mini: success
